In [ ]:
# Clone MTG-Jamendo repo
!git clone https://github.com/MTG/mtg-jamendo-dataset.git
%cd mtg-jamendo-dataset

# Install audio dependencies
!pip install pandas librosa yt-dlp Mutagen

Cloning into 'mtg-jamendo-dataset'...
remote: Enumerating objects: 1290, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 1290 (delta 246), reused 228 (delta 226), pack-reused 1030 (from 1)
Receiving objects: 100% (1290/1290), 42.37 MiB | 22.12 MiB/s, done.
Resolving deltas: 100% (792/792), done.
/content/mtg-jamendo-dataset
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.8 MB/s eta 0:00:00


In [ ]:
!mkdir -p /content/mtg_data/npy
%cd /content/mtg_data/npy
!wget -q https://cdn.freesound.org/mtg-jamendo/raw_30s/melspecs/raw_30s_melspecs-00.tar
!wget -q https://cdn.freesound.org/mtg-jamendo/raw_30s/melspecs/raw_30s_melspecs-01.tar
!wget -q https://cdn.freesound.org/mtg-jamendo/raw_30s/melspecs/raw_30s_melspecs-02.tar
!tar -xf raw_30s_melspecs-00.tar
!tar -xf raw_30s_melspecs-01.tar
!tar -xf raw_30s_melspecs-02.tar

/content/mtg_data/npy


In [ ]:
%cd /content/mtg-jamendo-dataset/scripts/baseline
!pip install fire -q
!python get_npy.py run '/content/mtg_data/npy'

/content/mtg-jamendo-dataset/scripts/baseline
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.8 MB/s eta 0:00:00


In [ ]:
%%writefile /content/mtg-jamendo-dataset/scripts/baseline/data_loader.py
import os
import numpy as np
import pickle
from torch.utils import data

FIXED_FRAMES = 1366  # ~29.1s at 12kHz/256-hop — matches the model's pooling layers exactly

class AudioFolder(data.Dataset):
    def __init__(self, root, subset, tr_val='train', split=0):
        self.trval = tr_val
        self.root = root
        fn = '../../data/splits/split-%d/%s_%s_dict.pickle' % (split, subset, tr_val)
        self.get_dictionary(fn)

    def __getitem__(self, index):
        fn = os.path.join(self.root, 'npy', self.dictionary[index]['path'][:-3]+'npy')
        audio = np.array(np.load(fn))
        audio = self._fix_length(audio)
        tags = self.dictionary[index]['tags']
        return audio.astype('float32'), tags.astype('float32'), self.dictionary[index]['path']

    def _fix_length(self, audio):
        n_frames = audio.shape[1]
        if n_frames >= FIXED_FRAMES:
            if self.trval == 'train':
                start = np.random.randint(0, n_frames - FIXED_FRAMES + 1)
            else:
                start = (n_frames - FIXED_FRAMES) // 2
            audio = audio[:, start:start + FIXED_FRAMES]
        else:
            pad_width = FIXED_FRAMES - n_frames
            audio = np.pad(audio, ((0, 0), (0, pad_width)), mode='constant')
        return audio

    def get_dictionary(self, fn):
        with open(fn, 'rb') as pf:
            dictionary = pickle.load(pf)
        self.dictionary = dictionary

    def __len__(self):
        return len(self.dictionary)


def get_audio_loader(root, subset, batch_size, tr_val='train', split=0, num_workers=0):
    data_loader = data.DataLoader(dataset=AudioFolder(root, subset, tr_val, split),
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=num_workers)
    return data_loader

Overwriting /content/mtg-jamendo-dataset/scripts/baseline/data_loader.py


In [ ]:
import re
path = '/content/mtg-jamendo-dataset/scripts/baseline/solver.py'
with open(path) as f:
    content = f.read()
content = content.replace(
    "for x, y in self.valid_loader:",
    "for x, y, _ in self.valid_loader:"
)
with open(path, 'w') as f:
    f.write(content)

In [ ]:
import re
path = '/content/mtg-jamendo-dataset/scripts/baseline/solver.py'
with open(path) as f:
    content = f.read()

old = """    def get_auc(self, prd_array, gt_array):
        roc_aucs = metrics.roc_auc_score(gt_array, prd_array, average='macro')
        pr_aucs = metrics.average_precision_score(gt_array, prd_array, average='macro')
        print('roc_auc: %.4f' % roc_aucs)
        print('pr_auc: %.4f' % pr_aucs)
        roc_auc_all = metrics.roc_auc_score(gt_array, prd_array, average=None)
        pr_auc_all = metrics.average_precision_score(gt_array, prd_array, average=None)
        for i in range(self.num_class):
            print('%s \\t\\t %.4f , %.4f' % (self.tag_list[i], roc_auc_all[i], pr_auc_all[i]))
        return roc_aucs, pr_aucs, roc_auc_all, pr_auc_all"""

new = """    def get_auc(self, prd_array, gt_array):
        roc_auc_all = np.full(self.num_class, np.nan)
        pr_auc_all = np.full(self.num_class, np.nan)
        for i in range(self.num_class):
            if len(np.unique(gt_array[:, i])) < 2:
                continue  # undefined for this tag in this split -- skip, don't poison the average
            roc_auc_all[i] = metrics.roc_auc_score(gt_array[:, i], prd_array[:, i])
            pr_auc_all[i] = metrics.average_precision_score(gt_array[:, i], prd_array[:, i])
        roc_aucs = np.nanmean(roc_auc_all)
        pr_aucs = np.nanmean(pr_auc_all)
        print('roc_auc: %.4f' % roc_aucs)
        print('pr_auc: %.4f' % pr_aucs)
        for i in range(self.num_class):
            print('%s \\t\\t %.4f , %.4f' % (self.tag_list[i], roc_auc_all[i], pr_auc_all[i]))
        return roc_aucs, pr_aucs, roc_auc_all, pr_auc_all"""

content = content.replace(old, new)
with open(path, 'w') as f:
    f.write(content)

In [ ]:
!grep -n "nanmean\|average='macro'" /content/mtg-jamendo-dataset/scripts/baseline/solver.py

172:        roc_aucs = metrics.roc_auc_score(gt_array, prd_array, average='macro')
173:        pr_aucs = metrics.average_precision_score(gt_array, prd_array, average='macro')


In [ ]:
import re

path = '/content/mtg-jamendo-dataset/scripts/baseline/solver.py'
with open(path) as f:
    content = f.read()

# Match the whole get_auc method, from its def line up through its return statement
pattern = re.compile(
    r"def get_auc\(self, prd_array, gt_array\):.*?return roc_aucs, pr_aucs, roc_auc_all, pr_auc_all",
    re.DOTALL
)

replacement = """def get_auc(self, prd_array, gt_array):
        roc_auc_all = np.full(self.num_class, np.nan)
        pr_auc_all = np.full(self.num_class, np.nan)
        for i in range(self.num_class):
            if len(np.unique(gt_array[:, i])) < 2:
                continue
            roc_auc_all[i] = metrics.roc_auc_score(gt_array[:, i], prd_array[:, i])
            pr_auc_all[i] = metrics.average_precision_score(gt_array[:, i], prd_array[:, i])
        roc_aucs = np.nanmean(roc_auc_all)
        pr_aucs = np.nanmean(pr_auc_all)
        print('roc_auc: %.4f' % roc_aucs)
        print('pr_auc: %.4f' % pr_aucs)
        for i in range(self.num_class):
            print('%s \\t\\t %.4f , %.4f' % (self.tag_list[i], roc_auc_all[i], pr_auc_all[i]))
        return roc_aucs, pr_aucs, roc_auc_all, pr_auc_all"""

new_content, n = pattern.subn(replacement, content)
print(f"Replacements made: {n}")  # this MUST print 1 — if it prints 0, the patch did not apply

with open(path, 'w') as f:
    f.write(new_content)

Replacements made: 1


In [ ]:
!grep -n "nanmean" /content/mtg-jamendo-dataset/scripts/baseline/solver.py

176:        roc_aucs = np.nanmean(roc_auc_all)
177:        pr_aucs = np.nanmean(pr_auc_all)


In [ ]:
import re

path = '/content/mtg-jamendo-dataset/scripts/baseline/solver.py'
with open(path) as f:
    content = f.read()

old = """    def get_auc(self, prd_array, gt_array):
        roc_auc_all = np.full(self.num_class, np.nan)"""

new = """    def get_auc(self, prd_array, gt_array):
        prd_array = np.array(prd_array)
        gt_array = np.array(gt_array)
        roc_auc_all = np.full(self.num_class, np.nan)"""

new_content, n = content.replace(old, new), content.count(old)
print(f"Replacements made: {n}")  # must print 1

with open(path, 'w') as f:
    f.write(content.replace(old, new))

Replacements made: 1


In [ ]:
%cd /content/mtg-jamendo-dataset/scripts/baseline
!python main.py --mode TRAIN --subset genre --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models' --batch_size 32

Streaming output truncated to the last 5000 lines.
genre---classical 		 0.9058 , 0.5079
genre---classicrock 		 nan , nan
genre---club 		 0.3392 , 0.0044
genre---contemporary 		 0.6186 , 0.0225
genre---country 		 0.7595 , 0.0270
genre---dance 		 0.8422 , 0.1705
genre---darkambient 		 nan , nan
genre---darkwave 		 0.4490 , 0.0101
genre---deephouse 		 nan , nan
genre---disco 		 0.4667 , 0.0108
genre---downtempo 		 0.6143 , 0.0539
genre---drumnbass 		 0.3065 , 0.0066
genre---dub 		 0.1228 , 0.0033
genre---dubstep 		 0.7275 , 0.0351
genre---easylistening 		 0.7570 , 0.2305
genre---edm 		 nan , nan
genre---electronic 		 0.7381 , 0.4949
genre---electronica 		 0.9064 , 0.0303
genre---electropop 		 0.6895 , 0.0225
genre---ethno 		 0.4120 , 0.0073
genre---eurodance 		 nan , nan
genre---experimental 		 0.5842 , 0.0915
genre---folk 		 0.8363 , 0.2389
genre---funk 		 0.8246 , 0.1384
genre---fusion 		 0.6280 , 0.0297
genre---groove 		 nan , nan
genre---grunge 		 0.9137 , 0.2340
genre---hard 		 0.555

In [ ]:
%cd /content/mtg-jamendo-dataset/scripts/baseline
!python main.py --mode TEST --subset genre --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models'

/content/mtg-jamendo-dataset/scripts/baseline
Namespace(batch_size=32, mode='TEST', model_save_path='./my_models', audio_path='/content/mtg_data', split=0, subset='genre')
[2026-07-31 13:00:23] Iter [10/11] test loss: 0.0880 Elapsed: 0:00:08.200401
roc_auc: 0.6916
pr_auc: 0.1373
genre---60s 		 nan , nan
genre---70s 		 0.7058 , 0.0355
genre---80s 		 0.6484 , 0.0258
genre---90s 		 0.7713 , 0.0250
genre---acidjazz 		 nan , nan
genre---alternative 		 0.7293 , 0.0752
genre---alternativerock 		 nan , nan
genre---ambient 		 0.8417 , 0.5044
genre---atmospheric 		 0.5863 , 0.0312
genre---blues 		 0.7385 , 0.0568
genre---bluesrock 		 0.5250 , 0.0166
genre---bossanova 		 nan , nan
genre---breakbeat 		 0.7500 , 0.0685
genre---celtic 		 0.3568 , 0.0088
genre---chanson 		 0.9883 , 0.2000
genre---chillout 		 0.7368 , 0.2100
genre---choir 		 nan , nan
genre---classical 		 0.9321 , 0.5333
genre---classicrock 		 0.4912 , 0.0084
genre---club 		 0.6260 , 0.0247
genre---contemporary 		 nan , nan
genre---co

In [ ]:
!python get_npy.py run '/content/mtg_data/npy'
!python main.py --mode TRAIN --subset instrument --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models_instrument' --batch_size 32
!python main.py --mode TEST --subset instrument --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models_instrument'

Streaming output truncated to the last 5000 lines.
instrument---pipeorgan 		 nan , nan
instrument---rhodes 		 0.7539 , 0.1656
instrument---sampler 		 0.8213 , 0.0620
instrument---saxophone 		 0.7327 , 0.1412
instrument---strings 		 0.4988 , 0.1009
instrument---synthesizer 		 0.7379 , 0.4889
instrument---trombone 		 0.6641 , 0.1522
instrument---trumpet 		 0.8246 , 0.5109
instrument---viola 		 0.5092 , 0.0584
instrument---violin 		 0.6286 , 0.3282
instrument---voice 		 0.6525 , 0.2374
[2026-07-31 13:27:17] Epoch [386/500] Iter [10/14] train loss: 0.2424 Elapsed: 0:25:14.003225
roc_auc: 0.6619
pr_auc: 0.1987
instrument---accordion 		 0.5061 , 0.0352
instrument---acousticbassguitar 		 0.6864 , 0.0455
instrument---acousticguitar 		 0.7196 , 0.1042
instrument---bass 		 0.7622 , 0.5066
instrument---beat 		 0.9699 , 0.5055
instrument---bell 		 0.5706 , 0.0492
instrument---bongo 		 0.6552 , 0.0688
instrument---brass 		 0.5078 , 0.0500
instrument---cello 		 0.6027 , 0.1871
instrument---clarinet 

In [ ]:
%cd /content/mtg-jamendo-dataset/scripts/baseline
!python main.py --mode TRAIN --subset genre --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models_bs16' --batch_size 16
!python main.py --mode TEST --subset genre --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models_bs16'

Streaming output truncated to the last 5000 lines.
genre---dub 		 0.1637 , 0.0035
genre---dubstep 		 0.9098 , 0.0708
genre---easylistening 		 0.7430 , 0.2085
genre---edm 		 nan , nan
genre---electronic 		 0.7402 , 0.5039
genre---electronica 		 0.2895 , 0.0041
genre---electropop 		 0.6785 , 0.0222
genre---ethno 		 0.4076 , 0.0090
genre---eurodance 		 nan , nan
genre---experimental 		 0.6067 , 0.0927
genre---folk 		 0.8565 , 0.3409
genre---funk 		 0.8356 , 0.1941
genre---fusion 		 0.5582 , 0.0276
genre---groove 		 nan , nan
genre---grunge 		 0.9392 , 0.1581
genre---hard 		 0.3626 , 0.0046
genre---hardrock 		 0.9115 , 0.0844
genre---hiphop 		 0.8507 , 0.4540
genre---house 		 0.8207 , 0.0520
genre---idm 		 0.8275 , 0.0167
genre---improvisation 		 0.5994 , 0.0072
genre---indie 		 0.6827 , 0.1776
genre---industrial 		 0.8097 , 0.0894
genre---instrumentalpop 		 0.7126 , 0.0712
genre---instrumentalrock 		 0.8038 , 0.0347
genre---jazz 		 0.7361 , 0.1506
genre---jazzfusion 		 nan , nan
genre---l

In [ ]:
import torch
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [ ]:
%cd /content/mtg-jamendo-dataset/scripts/baseline
!python main.py --mode TRAIN --subset genre --split 0 \
  --audio_path '/content/mtg_data' --model_save_path './my_models_bs64' --batch_size 64

/content/mtg-jamendo-dataset/scripts/baseline
Namespace(batch_size=64, mode='TRAIN', model_save_path='./my_models_bs64', audio_path='/content/mtg_data', split=0, subset='genre')
[2026-07-31 15:08:45] Epoch [1/500] Iter [10/16] train loss: 0.7149 Elapsed: 0:00:05.001031
Traceback (most recent call last):
  File "/content/mtg-jamendo-dataset/scripts/baseline/main.py", line 55, in <module>
    main(config)
  File "/content/mtg-jamendo-dataset/scripts/baseline/main.py", line 28, in main
    solver.train()
  File "/content/mtg-jamendo-dataset/scripts/baseline/solver.py", line 109, in train
    roc_auc, _ = self._validation(start_t, epoch)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/mtg-jamendo-dataset/scripts/baseline/solver.py", line 138, in _validation
    out = self.model(x)
          ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^